# BaCP across four models — dense, then every BaCP cell

Order: **resnet34 → vgg11 → resnet50 → vgg19**. For each model, the dense checkpoint first (BaCP resolves its starting weights from it), then every BaCP cell: **3 pruners × 4 sparsities = 12 per model**.

| | |
|---|---|
| pruners | magnitude, snip, wanda (all globally thresholded) |
| sparsities | 0.95, 0.97, 0.99, 0.999 |
| BaCP LR | 0.1 uniform, VGG-19 included (bf16 + grad-clip 10.0 now address the instability that forced 0.05) |
| tau / lambdas | 0.15 / 0.25 each |
| scope | classifier pruned; projection head never pruned |

Everything already recorded is skipped, so this is safe to re-run and safe to interrupt. `results.csv` is rewritten after every cell.

I.P. baselines are **not** run here — they come after, per the requested order.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')
import nb_common as nb
info = nb.setup()

## Sanity check — read before training

Protocol, what will run vs skip, and the dataset. Any FAIL stops the notebook before a GPU-second is spent.

In [ ]:
MODELS     = ['resnet34', 'vgg11', 'resnet50', 'vgg19']
PRUNERS    = ['magnitude', 'snip', 'wanda']
SPARSITIES = (0.95, 0.97, 0.99, 0.999)
SEED, GPU  = 1, 0

plan = []
for m in MODELS:
    plan.append(nb.make_cell(m, 'dense', seed=SEED))
    for p in PRUNERS:
        for s in SPARSITIES:
            plan.append(nb.make_cell(m, 'bacp', seed=SEED, pruner=p, sparsity=s))

print(f'{len(plan)} cells planned: {len(MODELS)} dense + '
      f'{len(MODELS)*len(PRUNERS)*len(SPARSITIES)} bacp')
assert nb.sanity_check(plan), 'sanity check failed'

## Run

Streams per-batch progress. Each model finishes its dense cell before its BaCP cells start, because BaCP loads that checkpoint.

In [ ]:
for cell in plan:
    nb.run(cell, gpu=GPU)
    nb.update_results_csv()

## Results so far

In [ ]:
import json, glob, os
root = os.environ['BACP_RESULTS_DIR']
rows = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    r = json.load(open(f, encoding='utf-8'))
    k = r.get('experiment_group') or ''
    if '.smoke' in k or r.get('status') != 'ok':
        continue
    rows[k] = r.get('test_acc_pct')

print(f'{"model":<10}{"pruner":<11}' + ''.join(f'{s:>9}' for s in SPARSITIES))
for m in MODELS:
    d = rows.get(f'static.dense.{m}.cifar10.dense.seed{SEED}')
    print(f'{m:<10}{"dense":<11}' + (f'{d:>9.2f}' if d else f'{"-":>9}'))
    for p in PRUNERS:
        line = f'{"":<10}{p:<11}'
        for s in SPARSITIES:
            v = rows.get(f'static.bacp.{m}.cifar10.s{s}.{p}.seed{SEED}')
            line += f'{v:>9.2f}' if v else f'{"-":>9}'
        print(line)